# 01 — Model Inspection & Iteration

This notebook runs a single modeling iteration and inspects the results.

**Workflow**: Run → inspect topics → check outlier rates → decide whether to iterate.

**Architecture**: Per-outlet BERTopic models (pre-trained, tuned to each corpus size)
→ `merge_models()` → unified topic space → per-outlet distributions.

**Why merged models?** A single global model lets large outlets (Tagesschau ~6k)
dominate topic discovery, drowning out niche topics from small outlets (Antispiegel ~565).
Per-outlet models discover each outlet's topics independently; the merge creates a
*union* of all discovered topics — exactly what we need to detect agenda distortion.

In [ ]:
import os
import sys
from pathlib import Path

# Find project root
def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("No .git found")

PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "agenda_distortion"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add experiment and BERTopic modules to path
for p in [str(EXPERIMENT_DIR), str(PROJECT_ROOT / "1a_BERTopic")]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

print(f"Project root:   {PROJECT_ROOT}")
print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Output dir:     {OUTPUT_DIR}")

## Iteration Parameters

Change these to start a new iteration. Document why in the iteration log.

In [ ]:
from modeling import IterationParams

ITERATION_ID = "v1"

params = IterationParams(
    min_similarity=0.7,              # merge threshold (lower = fewer topics)
    outlier_strategy="c-tf-idf",     # uniform outlier reduction
    outlier_threshold=0.10,          # reassignment threshold
    umap_n_neighbors=10,             # for 2D projection only
    umap_min_dist=0.0,
    random_state=42,
    min_articles_for_coverage=10,    # minimum for breadth metric
)

print(f"Iteration: {ITERATION_ID}")
print(f"Parameters: {params.to_dict()}")

## Run Iteration

This loads pre-trained per-outlet models, merges them, assigns all documents
to the merged topic space, and applies uniform outlier reduction.

In [ ]:
from modeling import run_iteration, save_iteration

result = run_iteration(PROJECT_ROOT, ITERATION_ID, params)

# Save for downstream notebooks
save_iteration(result, OUTPUT_DIR)

merged_articles = result.merged_articles
merged_topic_info = result.merged_topic_info

print(f"\n{'='*60}")
print(f"  Iteration '{ITERATION_ID}' complete")
print(f"  Merged topics: {result.n_topics}")
print(f"  Total articles: {len(merged_articles):,}")
print(f"  Duration: {result.duration_seconds:.0f}s")
print(f"{'='*60}")

## Corpus Overview

Check class imbalance and outlier rates per outlet.
Outlier rates should be comparable across outlets — if one is much higher,
its topic distribution is less reliable.

In [ ]:
import pandas as pd
from IPython.display import display

summary_df = pd.DataFrame(result.outlet_summary).T
summary_df.index.name = "Outlet"
summary_df = summary_df.sort_values("n_articles", ascending=False)

print("Corpus composition & outlier rates:")
display(summary_df)

# Imbalance warning
sizes = summary_df["n_articles"]
ratio = sizes.max() / sizes.min()
print(f"\nSize ratio (largest/smallest): {ratio:.1f}x")
if ratio > 5:
    print("⚠ Large imbalance — rely on size-controlled metrics (JSD, Spearman, Top-K overlap)")

# Outlier rate check
max_outlier = summary_df["outlier_rate"].max()
min_outlier = summary_df["outlier_rate"].min()
print(f"Outlier rate range: {min_outlier:.1%} – {max_outlier:.1%}")
if max_outlier - min_outlier > 0.15:
    print("⚠ Large outlier rate variation — topic distributions may not be comparable")

## Topic Inspection

Check: are the top topics interpretable? Any garbage/noise clusters?

In [ ]:
# Top 20 merged topics
top_topics = (
    merged_topic_info
    .loc[merged_topic_info["Topic"] != -1]
    .sort_values("Count", ascending=False)
    .head(20)
    [["DisplayTopic", "DisplayLabel", "Count"]]
)
display(top_topics)

In [ ]:
# Which outlets contribute to the top 10 topics?
top10_ids = (
    merged_topic_info
    .loc[merged_topic_info["Topic"] != -1]
    .nsmallest(10, "DisplayTopic")["Topic"]
    .tolist()
)

cross_tab = pd.crosstab(
    merged_articles.loc[merged_articles["merged_topic"].isin(top10_ids), "merged_display_label"],
    merged_articles.loc[merged_articles["merged_topic"].isin(top10_ids), "outlet_label"],
)
display(cross_tab)

## UMAP — Global Topic Landscape

In [ ]:
from merged_outlets_analysis import plot_merged_topic_umap, THESIS_COLORS

fig, ax = plot_merged_topic_umap(merged_articles, merged_topic_info, top_n=20)
fig.savefig(OUTPUT_DIR / ITERATION_ID / "umap_global.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Semantic Footprint Maps

One per outlet — shows where each outlet's articles cluster in the merged topic space.

In [ ]:
import matplotlib.pyplot as plt
from visualization import plot_all_outlet_umaps

figs = plot_all_outlet_umaps(
    merged_articles,
    merged_topic_info,
    save_dir=OUTPUT_DIR / ITERATION_ID / "footprints",
)
for fig in figs:
    plt.show()
    plt.close(fig)

## Decision Point

**Check before proceeding to notebook 02:**

- [ ] Top topics are semantically coherent (not noise/garbage)
- [ ] Outlier rates are comparable across outlets (< 15% difference)
- [ ] Topic count is reasonable (not too few, not fragmented)
- [ ] UMAP shows identifiable clusters, not a uniform blob

If any check fails → change parameters above and re-run.

If all pass → proceed to `02_h1_results.ipynb`.